### Clumping Metric

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from matplotlib.animation import FuncAnimation  # noqa: F401
from matplotlib.widgets import Slider  # noqa: F401
from IPython.display import HTML  # noqa: F401


def gaussian(x, amp, mu, sigma):
    x = np.asarray(x)
    return amp * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def random_points_on_sphere(N):
    """Uniform random points on the unit sphere."""
    phi = 2 * np.pi * np.random.rand(N)
    mu = 2 * np.random.rand(N) - 1

    x = np.sqrt(1 - mu**2) * np.cos(phi)
    y = np.sqrt(1 - mu**2) * np.sin(phi)
    z = mu

    r_hat = np.vstack((x, y, z)).T
    return r_hat, mu, phi


def random_points_in_sphere(N, r=1.0):
    """Uniform random points inside a sphere of radius r."""
    rand_r = r * np.cbrt(np.random.rand(N))
    theta = 2 * np.pi * np.random.rand(N)
    phi = np.arccos(2 * np.random.rand(N) - 1)

    x = rand_r * np.sin(phi) * np.cos(theta)
    y = rand_r * np.sin(phi) * np.sin(theta)
    z = rand_r * np.cos(phi)

    return np.vstack((x, y, z)).T


def fibonacci_sphere(N):
    """Deterministic near-uniform points on sphere (Fibonacci spiral)."""
    i = np.arange(N)
    golden_angle = np.pi * (3.0 - np.sqrt(5.0))

    y = 1.0 - 2.0 * (i + 0.5) / N
    r = np.sqrt(1.0 - y * y)
    theta = golden_angle * i

    x = r * np.cos(theta)
    z = r * np.sin(theta)

    return np.vstack((x, y, z)).T


def R_x(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def R_y(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def smooth_profile(v_los, weights, sigma=0.08, v_grid=None):
    """Gaussian smooth a weighted line profile on the provided velocity grid."""
    if v_grid is None:
        raise ValueError("v_grid must be provided")

    v_los = np.asarray(v_los)
    weights = np.asarray(weights)

    diff = v_grid[:, None] - v_los[None, :]
    kernel = np.exp(-0.5 * (diff / sigma) ** 2)
    profile = kernel @ weights

    m = np.max(profile)
    if m > 0:
        profile = profile / m

    return profile


#### Old

In [ ]:
# --- Sphere geometry (parametric grid) ---
u = np.linspace(0, 2 * np.pi, 100)
v = np.linspace(0, np.pi, 100)

r = 1
x = r * np.outer(np.cos(u), np.sin(v))
y = r * np.outer(np.sin(u), np.sin(v))
z = r * np.outer(np.ones_like(u), np.cos(v))

# --- Colormap rotation around x-axis ---
theta_deg = 135
theta = np.radians(theta_deg)

# Apply rotation
y_rot = y * np.cos(theta) - z * np.sin(theta)
z_rot = y * np.sin(theta) + z * np.cos(theta)

# Normalize rotated z for colormap
cmap_values = (z_rot - z_rot.min()) / (z_rot.max() - z_rot.min())

In [ ]:
# --- Function to draw a thick 3D arrow (cylinder + cone) ---
def draw_3d_arrow(ax, start, direction, length=0.6, radius=0.05, color='k'):
    # Normalize direction
    direction = direction / np.linalg.norm(direction)
    
    # Shaft (cylinder)
    shaft_length = 0.75 * length
    tip_length = 0.25 * length

    # Cylinder parameterization
    t = np.linspace(0, shaft_length, 20)
    theta = np.linspace(0, 2*np.pi, 20)
    t, theta = np.meshgrid(t, theta)

    # Local cylinder coordinates
    xc = radius * np.cos(theta)
    yc = radius * np.sin(theta)
    zc = t

    # Build rotation matrix to align cylinder with direction vector
    z_axis = np.array([0, 0, 1])
    v = np.cross(z_axis, direction)
    c = np.dot(z_axis, direction)
    if np.linalg.norm(v) < 1e-8:
        R = np.eye(3)
    else:
        vx = np.array([[0, -v[2], v[1]],
                       [v[2], 0, -v[0]],
                       [-v[1], v[0], 0]])
        R = np.eye(3) + vx + vx @ vx * (1 / (1 + c))

    # Rotate and translate cylinder
    xyz = np.stack([xc, yc, zc], axis=-1)
    xyz_rot = xyz @ R.T + start

    ax.plot_surface(
        xyz_rot[...,0], xyz_rot[...,1], xyz_rot[...,2],
        color=color, shade=True
    )

    # Cone tip
    t2 = np.linspace(0, tip_length, 20)
    theta2 = np.linspace(0, 2*np.pi, 20)
    t2, theta2 = np.meshgrid(t2, theta2)

    rc = radius * (1 - t2 / tip_length)
    xc2 = rc * np.cos(theta2)
    yc2 = rc * np.sin(theta2)
    zc2 = t2 + shaft_length

    xyz2 = np.stack([xc2, yc2, zc2], axis=-1)
    xyz2_rot = xyz2 @ R.T + start

    ax.plot_surface(
        xyz2_rot[...,0], xyz2_rot[...,1], xyz2_rot[...,2],
        color=color, shade=True
    )


In [ ]:
# --- Figure with two equal-sized panels ---
fig = plt.figure(figsize=(12, 6))

# Left: 3D sphere panel
ax = fig.add_subplot(1, 2, 1, projection='3d')

# --- Render sphere with rotated colormap ---
ax.plot_surface(
    x, y, z,
    facecolors=plt.cm.twilight(cmap_values),
    rstride=1, cstride=1,
    linewidth=0, antialiased=False
)

ax.set_box_aspect([1, 1, 1])
ax.set_title("3D Sphere Surface")

# --- Arrows from sphere surface toward +z, -x, -y ---

arrow_length = 0.6   # length of arrows
sphere_radius = 1.0  # same as r

# 1. Arrow toward +z (start at north pole)
start_z = np.array([0, 0, sphere_radius])
ax.quiver(
    start_z[0], start_z[1], start_z[2],
    0, 0, arrow_length,
    color='k', linewidth=2
)

# 2. Arrow toward -x (start at +x surface)
start_x = np.array([sphere_radius, 0, 0])
ax.quiver(
    start_x[0], start_x[1], start_x[2],
    arrow_length, 0, 0,
    color='k', linewidth=2
)

# 3. Arrow toward -y (start at +y surface)
start_y = np.array([-sphere_radius*0.6, -sphere_radius*0.8, 0])
ax.quiver(
    start_y[0], start_y[1], start_y[2],
    -arrow_length*0.6, -arrow_length*0.8, 0,
    color='k', linewidth=2
)


# --- Right-side 2D panel ---
# Right: 2D panel
ax2 = fig.add_subplot(1, 2, 2)
ax2.set_title("2D Panel")
ax2.plot([1142, 1142, 1172, 1172], [0, 4, 4, 0], linewidth=2)
ax2.set_xlabel("Energy (keV)")
ax2.set_ylabel("Flux (10$^{-6}$ ph cm$^{-2}$ s$^{-1}$)")
ax2.set_xlim([1140, 1175])
ax2.set_ylim([-0.5, 5])

# --- Inset orientation triad (bottom-left of figure) ---
ax_inset = fig.add_axes([0.1, 0.1, 0.15, 0.15], projection='3d')

L = 1.0
ax_inset.quiver(0, 0, 0, L, 0, 0, color='g')  # X axis
ax_inset.quiver(0, 0, 0, 0, L, 0, color='b')  # Y axis
ax_inset.quiver(0, 0, 0, 0, 0, L, color='r')  # Z axis

# Clean inset appearance
ax_inset.xaxis.pane.set_visible(False)
ax_inset.yaxis.pane.set_visible(False)
ax_inset.zaxis.pane.set_visible(False)
ax_inset.grid(False)

ax_inset.set_xlim([0, L])
ax_inset.set_ylim([0, L])
ax_inset.set_zlim([0, L])
ax_inset.set_xticks([])
ax_inset.set_yticks([])
ax_inset.set_zticks([])
ax_inset.set_box_aspect([1, 1, 1])
ax_inset.set_facecolor("white")

plt.show()

In [ ]:
# --- Left panel: 3D random points inside a sphere ---
fig = plt.figure(figsize=(12, 6))
ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.set_title("Random Points in Spherical Volume")
ax.set_box_aspect([1, 1, 1])

# Number of points
N = 100
r = 1.0

# Uniform random distribution inside a sphere:
# radius must be cubic-rooted for uniform volume density
rand_r = r * np.cbrt(np.random.rand(N))
theta = 2 * np.pi * np.random.rand(N)
phi = np.arccos(2 * np.random.rand(N) - 1)

# Convert spherical → Cartesian
x_pts = rand_r * np.sin(phi) * np.cos(theta)
y_pts = rand_r * np.sin(phi) * np.sin(theta)
z_pts = rand_r * np.cos(phi)

# Plot the points
ax.scatter(x_pts, y_pts, z_pts, s=40, c='purple', alpha=0.8)

# --- Add short random-direction arrows from each point ---

arrow_length = 0.15  # short arrows so they don't clutter

# Generate random directions for each point
rand_dirs = np.random.normal(size=(N, 3))
rand_dirs /= np.linalg.norm(rand_dirs, axis=1)[:, None]  # normalize each vector

# Plot arrows
ax.quiver(
    x_pts, y_pts, z_pts,               # starting points
    rand_dirs[:,0], rand_dirs[:,1], rand_dirs[:,2],   # directions
    length=arrow_length,
    normalize=True,
    color='black',
    linewidth=1.2
)


# --- Right-side 2D panel ---
# Right: 2D panel
ax2 = fig.add_subplot(1, 2, 2)
ax2.set_title("2D Panel")
x1 = np.linspace(1142, 1172, 91)
ax2.plot(x1, gaussian(x1, 60, 1157, 5), linewidth=2)
ax2.set_xlabel("Energy (keV)")
ax2.set_ylabel("Flux (10$^{-6}$ ph cm$^{-2}$ s$^{-1}$)")
ax2.set_xlim([1140, 1175])
ax2.set_ylim([-0.5, 5])

# --- Inset orientation triad (bottom-left of figure) ---
ax_inset = fig.add_axes([0.1, 0.1, 0.15, 0.15], projection='3d')

L = 1.0
ax_inset.quiver(0, 0, 0, L, 0, 0, color='g')  # X axis
ax_inset.quiver(0, 0, 0, 0, L, 0, color='b')  # Y axis
ax_inset.quiver(0, 0, 0, 0, 0, L, color='r')  # Z axis

# Clean inset appearance
ax_inset.xaxis.pane.set_visible(False)
ax_inset.yaxis.pane.set_visible(False)
ax_inset.zaxis.pane.set_visible(False)
ax_inset.grid(False)

ax_inset.set_xlim([0, L])
ax_inset.set_ylim([0, L])
ax_inset.set_zlim([0, L])
ax_inset.set_xticks([])
ax_inset.set_yticks([])
ax_inset.set_zticks([])
ax_inset.set_box_aspect([1, 1, 1])
ax_inset.set_facecolor("white")

plt.show()

In [ ]:
# Creating 3D plot of a sphere with radial velocity vectors

# Step 1: Generate N random points on the sphere
N = 200
phi = np.random.uniform(0, 2 * np.pi, N)  # azimuthal angle
cos_theta = np.random.uniform(-1, 1, N)   # cosine of polar angle
theta = np.arccos(cos_theta)

# Step 2: Convert spherical to Cartesian coordinates
x = np.sin(theta) * np.cos(phi)
y = np.sin(theta) * np.sin(phi)
z = np.cos(theta)

# Step 3: Define velocity vectors (radial, normalized)
v = 1.0
vx = v * x
vy = v * y
vz = v * z

# Step 4-7: Create 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_box_aspect([1,1,1])

# Plot points
ax.scatter(x, y, z, color='blue', s=10, alpha=0.6, label='Surface Points')

# Plot velocity vectors
ax.quiver(x, y, z, vx, vy, vz, length=0.1, normalize=True, color='red', linewidth=0.5)

# Label axes
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Radial Velocity Vectors on a Sphere')

# Set equal aspect ratio
ax.set_xlim([-1.2, 1.2])
ax.set_ylim([-1.2, 1.2])
ax.set_zlim([-1.2, 1.2])

# Save figure
# output_path = "/mnt/data/sphere_radial_velocity_vectors.png"
# plt.savefig(output_path, dpi=300)
# plt.close()

print("3D plot of radial velocity vectors on a sphere saved as 'sphere_radial_velocity_vectors.png'")


In [ ]:
# Generating 3D plot of sphere with dipole-weighted velocity arrows

# Step 1: Generate N random points on the sphere
N = 300
phi = np.random.uniform(0, 2 * np.pi, N)
cos_theta = np.random.uniform(-1, 1, N)
theta = np.arccos(cos_theta)

# Step 2: Convert spherical to Cartesian coordinates
x = np.sin(theta) * np.cos(phi)
y = np.sin(theta) * np.sin(phi)
z = np.cos(theta)

# Step 3: Dipole axis along z-axis, so dipole weight is cos(theta) = z
w = z.copy()

# Step 4: Normalize weights to [0, 1]
w_min, w_max = w.min(), w.max()
w_norm = (w - w_min) / (w_max - w_min)

# Step 5: Define velocity vectors scaled by dipole weight
v = 1.0  # base speed
vx = v * x * w_norm
vy = v * y * w_norm
vz = v * z * w_norm

# Step 6-9: Create 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_title("Dipole-Weighted Radial Velocity Vectors on a Sphere")

# Scatter plot colored by dipole weight
sc = ax.scatter(x, y, z, c=w_norm, cmap='coolwarm', s=20)

# Quiver plot for velocity vectors
ax.quiver(x, y, z, vx, vy, vz, length=0.1, normalize=False, color='black', linewidth=0.5)

# Set equal aspect ratio
ax.set_box_aspect([1,1,1])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')

# Add colorbar
cbar = plt.colorbar(sc, ax=ax, shrink=0.6)
cbar.set_label('Dipole Weight (Normalized)')

# Save the figure
# output_path = "/mnt/data/dipole_weighted_velocity_sphere.png"
# plt.savefig(output_path, dpi=300, bbox_inches='tight')
# plt.close()

print("3D plot of dipole-weighted velocity vectors saved as 'dipole_weighted_velocity_sphere.png'")


In [ ]:
# -----------------------------
# Parameters
# -----------------------------
N = 800          # number of points on the sphere
v0 = 1.0         # radial speed (sets max |v_los|)
nbins = 40       # number of velocity bins
nframes = 120    # number of animation frames

# -----------------------------
# Generate random points on a sphere
# -----------------------------
# Uniform on sphere: phi uniform in [0, 2pi), cos(theta) uniform in [-1, 1]
phi = 2 * np.pi * np.random.rand(N)
mu = 2 * np.random.rand(N) - 1  # cos(theta)
theta = np.arccos(mu)

# Cartesian coordinates (unit vectors r_hat)
x = np.sqrt(1 - mu**2) * np.cos(phi)
y = np.sqrt(1 - mu**2) * np.sin(phi)
z = mu
r_hat = np.vstack((x, y, z)).T  # shape (N, 3)

# Line-of-sight direction: along +z
los = np.array([0.0, 0.0, 1.0])

# Line-of-sight velocities (for pure radial expansion)
v_los = v0 * (r_hat @ los)  # v0 * cos(theta_los)

# Velocity bin edges
vmin, vmax = -v0, v0
bins = np.linspace(vmin, vmax, nbins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

# -----------------------------
# Dipole axis rotation
# -----------------------------
def dipole_axis(phi_rot):
    """
    Dipole axis rotated by angle phi_rot around the y-axis.
    Start with axis along +z: (0, 0, 1).
    """
    # Rotation matrix around y-axis
    c = np.cos(phi_rot)
    s = np.sin(phi_rot)
    R = np.array([[ c, 0,  s],
                  [ 0, 1,  0],
                  [-s, 0,  c]])
    return R @ np.array([0.0, 0.0, 1.0])

# -----------------------------
# Set up figure
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 4))
line, = ax.plot([], [], lw=2)

ax.set_xlim(vmin, vmax)
ax.set_ylim(0, 1.1)  # will rescale with normalization
ax.set_xlabel(r"$v_{\rm los}$")
ax.set_ylabel("Normalized intensity")
ax.set_title("Doppler line profile with rotating dipole axis")

# -----------------------------
# Animation update function
# -----------------------------
def init():
    line.set_data([], [])
    return line,

def update(frame):
    # Angle of dipole axis for this frame
    phi_rot = 2 * np.pi * frame / nframes
    d_axis = dipole_axis(phi_rot)

    # Dipole weight: w = r_hat · d_axis
    w = r_hat @ d_axis

    # Optional: keep only positive part (e.g., emissivity ∝ max(w, 0))
    # Comment/uncomment depending on your physical model
    # w_eff = np.maximum(w, 0.0)
    # For a pure dipole (allowing negative weights), use:
    w_eff = w

    # Weighted histogram of v_los
    hist, _ = np.histogram(v_los, bins=bins, weights=w_eff)

    # Normalize profile for visualization
    if np.max(np.abs(hist)) > 0:
        hist_norm = hist / np.max(np.abs(hist))
    else:
        hist_norm = hist

    line.set_data(bin_centers, hist_norm)
    ax.set_title(
        "Doppler line profile\nDipole axis rotation angle = "
        f"{np.degrees(phi_rot):.1f}°"
    )
    return line,

anim = FuncAnimation(
    fig, update, frames=nframes, init_func=init,
    blit=True, interval=80, repeat=True
)

plt.tight_layout()
plt.show()

# If you want to save as MP4 (requires ffmpeg installed), uncomment:
anim.save("doppler_dipole_rotation.gif", writer="pillow", fps=30)


In [ ]:
# Generating 3D plot with velocity arrows scaled by quadrupole weighting

# Generate random points on a sphere
N = 300
phi = 2 * np.pi * np.random.rand(N)
mu = 2 * np.random.rand(N) - 1  # cos(theta)
theta = np.arccos(mu)

x = np.sqrt(1 - mu**2) * np.cos(phi)
y = np.sqrt(1 - mu**2) * np.sin(phi)
z = mu
r_hat = np.vstack((x, y, z)).T

# Quadrupole weighting: Y20 proportional to (3 cos^2(theta) - 1)
w = 3 * mu**2 - 1
w_norm = (w - w.min()) / (w.max() - w.min())  # normalize to [0,1]

# Velocity vectors scaled by quadrupole weight
vectors = r_hat * w_norm[:, None]

# Plot
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# Scatter points colored by quadrupole weight
sc = ax.scatter(x, y, z, c=w_norm, cmap='coolwarm', s=20)

# Quiver arrows
ax.quiver(x, y, z, vectors[:,0], vectors[:,1], vectors[:,2], length=0.2, normalize=False, color='black', linewidth=0.5)

ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('3D Sphere with Quadrupole-Weighted Velocity Arrows')
ax.set_box_aspect([1,1,1])
plt.colorbar(sc, label='Quadrupole Weight (normalized)')

# Save the plot
# output_path = "/mnt/data/quadrupole_velocity_plot.png"
# plt.savefig(output_path, dpi=300, bbox_inches='tight')
# plt.close()

print("Created 3D plot of quadrupole-weighted velocity vectors and saved as 'quadrupole_velocity_plot.png'")


In [ ]:
# -----------------------------
# Parameters
# -----------------------------
N = 800          # number of points on the sphere
v0 = 1.0         # radial speed (sets max |v_los|)
nbins = 50       # number of velocity bins
nframes = 120    # number of animation frames

# -----------------------------
# Generate random points on a sphere
# -----------------------------
phi = 2 * np.pi * np.random.rand(N)
mu = 2 * np.random.rand(N) - 1  # cos(theta)
theta = np.arccos(mu)

x = np.sqrt(1 - mu**2) * np.cos(phi)
y = np.sqrt(1 - mu**2) * np.sin(phi)
z = mu
r_hat = np.vstack((x, y, z)).T  # shape (N, 3)

# Line-of-sight direction: +z
los = np.array([0.0, 0.0, 1.0])

# Line-of-sight velocities
v_los = v0 * (r_hat @ los)

# Velocity bins
vmin, vmax = -v0, v0
bins = np.linspace(vmin, vmax, nbins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

# -----------------------------
# Quadrupole axis rotation
# -----------------------------
def quadrupole_axis(phi_rot):
    """Rotate quadrupole axis around y-axis by phi_rot."""
    c = np.cos(phi_rot)
    s = np.sin(phi_rot)
    R = np.array([[ c, 0,  s],
                  [ 0, 1,  0],
                  [-s, 0,  c]])
    return R @ np.array([0.0, 0.0, 1.0])

# -----------------------------
# Set up figure
# -----------------------------
fig, ax = plt.subplots(figsize=(6, 4))
line, = ax.plot([], [], lw=2)

ax.set_xlim(vmin, vmax)
ax.set_ylim(0, 1.1)
ax.set_xlabel(r"$v_{\rm los}$")
ax.set_ylabel("Normalized intensity")
ax.set_title("Quadrupole-weighted Doppler line profile")

# -----------------------------
# Animation update
# -----------------------------
def init():
    line.set_data([], [])
    return line,

def update(frame):
    phi_rot = 2 * np.pi * frame / nframes
    q_axis = quadrupole_axis(phi_rot)

    # Quadrupole weight: w = 3 (r·axis)^2 - 1
    dot = r_hat @ q_axis
    w = 3 * dot**2 - 1

    # Weighted histogram
    hist, _ = np.histogram(v_los, bins=bins, weights=w)

    # Normalize
    if np.max(np.abs(hist)) > 0:
        hist_norm = hist / np.max(np.abs(hist))
    else:
        hist_norm = hist

    line.set_data(bin_centers, hist_norm)
    ax.set_title(
        f"Quadrupole Doppler Profile\nRotation angle = {np.degrees(phi_rot):.1f}°"
    )
    return line,

anim = FuncAnimation(
    fig, update, frames=nframes, init_func=init,
    blit=True, interval=80, repeat=True
)

# plt.close()  # Prevent duplicate static image
# HTML(anim.to_jshtml())
anim.save("doppler_quadrupole_rotation.gif", writer="pillow", fps=30)


#### New

https://copilot.microsoft.com/chats/sWJT36YKrZ3UgzhH1CPaP

In [ ]:
# -----------------------------
# Parameters
# -----------------------------
N = 600
v0 = 1.0
nbins = 40
nframes = 120

# -----------------------------
# Generate random points on a sphere
# -----------------------------
phi = 2 * np.pi * np.random.rand(N)
mu = 2 * np.random.rand(N) - 1
theta = np.arccos(mu)

x = np.sqrt(1 - mu**2) * np.cos(phi)
y = np.sqrt(1 - mu**2) * np.sin(phi)
z = mu
r_hat = np.vstack((x, y, z)).T

# LOS direction
los = np.array([0, 0, 1])
v_los = v0 * (r_hat @ los)

# Histogram bins
bins = np.linspace(-v0, v0, nbins + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

# -----------------------------
# Quadrupole axis rotation
# -----------------------------
def quadrupole_axis(phi_rot):
    c, s = np.cos(phi_rot), np.sin(phi_rot)
    R = np.array([[ c, 0,  s],
                  [ 0, 1,  0],
                  [-s, 0,  c]])
    return R @ np.array([0, 0, 1])

# -----------------------------
# Figure setup
# -----------------------------
fig = plt.figure(figsize=(12, 5))

ax_sphere = fig.add_subplot(121, projection='3d')
ax_profile = fig.add_subplot(122)

# Sphere scatter (updated each frame)
scatter = ax_sphere.scatter([], [], [], c=[], cmap="Blues", s=20)

# Quiver placeholder
quiver = None

# Line profile
line, = ax_profile.plot([], [], lw=2)

ax_sphere.set_title("Quadrupole‑Weighted Sphere")
ax_sphere.set_box_aspect([1,1,1])
ax_sphere.set_xlim([-1,1])
ax_sphere.set_ylim([-1,1])
ax_sphere.set_zlim([-1,1])

ax_profile.set_xlim([-v0, v0])
ax_profile.set_ylim([0, 1.1])
ax_profile.set_xlabel("v_los")
ax_profile.set_ylabel("Normalized intensity")

# -----------------------------
# Animation update
# -----------------------------
def update(frame):
    global quiver

    phi_rot = 2 * np.pi * frame / nframes
    q_axis = quadrupole_axis(phi_rot)

    # Quadrupole weight
    dot = r_hat @ q_axis
    w = 3 * dot**2 - 1
    w_norm = (w - w.min()) / (w.max() - w.min())

    # Update sphere scatter
    scatter._offsets3d = (x, y, z)
    scatter.set_array(w_norm)

    # Update quiver
    if quiver:
        quiver.remove()

    vectors = r_hat * w_norm[:, None]
    quiver = ax_sphere.quiver(
        x, y, z,
        vectors[:,0], vectors[:,1], vectors[:,2],
        length=0.2, normalize=False, color="black", linewidth=0.5
    )

    # Doppler profile
    hist, _ = np.histogram(v_los, bins=bins, weights=w)
    hist_norm = hist / np.max(np.abs(hist)) if np.max(np.abs(hist)) > 0 else hist

    line.set_data(bin_centers, hist_norm)
    ax_profile.set_title(f"Quadrupole Doppler Profile\nRotation = {np.degrees(phi_rot):.1f}°")

    return scatter, quiver, line

anim = FuncAnimation(fig, update, frames=nframes, interval=80, blit=False)

# plt.close()
# HTML(anim.to_jshtml())
anim.save("crazy.gif", writer="pillow", fps=10)


In [ ]:
# -----------------------------
# Parameters
# -----------------------------
N = 600
v0 = 1.0
nframes = 120
sigma = 0.08   # Gaussian width in velocity units
nv = 400       # resolution of velocity grid

# Velocity grid for smoothed profile
v_grid = np.linspace(-v0, v0, nv)

# -----------------------------
# Generate random points on a sphere
# -----------------------------
phi = 2 * np.pi * np.random.rand(N)
mu = 2 * np.random.rand(N) - 1
theta = np.arccos(mu)

x = np.sqrt(1 - mu**2) * np.cos(phi)
y = np.sqrt(1 - mu**2) * np.sin(phi)
z = mu
r_hat = np.vstack((x, y, z)).T

# Line-of-sight direction
los = np.array([0, 0, 1])
v_los = v0 * (r_hat @ los)

# -----------------------------
# Quadrupole axis rotation
# -----------------------------
def quadrupole_axis(phi_rot):
    c, s = np.cos(phi_rot), np.sin(phi_rot)
    R = np.array([[ c, 0,  s],
                  [ 0, 1,  0],
                  [-s, 0,  c]])
    return R @ np.array([0, 0, 1])

# -----------------------------
# Gaussian smoothing per point
# -----------------------------
def smooth_profile(v_los, weights):
    # Each point contributes a Gaussian
    diff = v_grid[:, None] - v_los[None, :]
    kernel = np.exp(-0.5 * (diff / sigma)**2)
    profile = kernel @ weights
    return profile / np.max(profile)

# -----------------------------
# Figure setup
# -----------------------------
fig = plt.figure(figsize=(12, 5))

ax_sphere = fig.add_subplot(121, projection='3d')
ax_profile = fig.add_subplot(122)

# Sphere scatter
scatter = ax_sphere.scatter([], [], [], c=[], cmap="Blues", s=20)

# Velocity arrows
quiver = None

# Orientation triad at bottom-left
triad_origin = np.array([-1.2, -1.2, -1.2])
triad_dirs = np.array([[0.5, 0, 0], [0, 0.5, 0], [0, 0, 0.5]])
triad_colors = ["red", "green", "blue"]
triad = ax_sphere.quiver(
    triad_origin[0], triad_origin[1], triad_origin[2],
    triad_dirs[:,0], triad_dirs[:,1], triad_dirs[:,2],
    length=0.3, normalize=False, colors=triad_colors, linewidth=2
)

ax_sphere.text(1.1, 0, 0, "x", color="red")
ax_sphere.text(0, 1.1, 0, "y", color="green")
ax_sphere.text(0, 0, 1.1, "z (view)", color="blue")

# Line profile
line, = ax_profile.plot([], [], lw=2)

ax_sphere.set_title("Quadrupole-Weighted Sphere")
ax_sphere.set_box_aspect([1,1,1])
ax_sphere.set_xlim([-1,1])
ax_sphere.set_ylim([-1,1])
ax_sphere.set_zlim([-1,1])

ax_profile.set_xlim([-v0, v0])
ax_profile.set_ylim([0, 1.1])
ax_profile.set_xlabel("v_los")
ax_profile.set_ylabel("Normalized intensity")

# -----------------------------
# Animation update
# -----------------------------
def update(frame):
    global quiver

    phi_rot = 2 * np.pi * frame / nframes
    q_axis = quadrupole_axis(phi_rot)

    # Quadrupole weight
    dot = r_hat @ q_axis
    w = 3 * dot**2 - 1
    w_norm = (w - w.min()) / (w.max() - w.min())

    # Update sphere scatter
    scatter._offsets3d = (x, y, z)
    scatter.set_array(w_norm)

    # Update velocity arrows
    if quiver:
        quiver.remove()

    vectors = r_hat * w_norm[:, None]
    quiver = ax_sphere.quiver(
        x, y, z,
        vectors[:,0], vectors[:,1], vectors[:,2],
        length=0.2, normalize=False, color="black", linewidth=0.5
    )

    # Smoothed Doppler profile
    profile = smooth_profile(v_los, w)

    line.set_data(v_grid, profile)
    ax_profile.set_title(
        f"Gaussian-Smoothed Line Profile\nRotation = {np.degrees(phi_rot):.1f}°"
    )

    return scatter, quiver, line, triad

anim = FuncAnimation(fig, update, frames=nframes, interval=80, blit=False)

# plt.close()
# HTML(anim.to_jshtml())
anim.save("crazy.gif", writer="pillow", fps=10)


In [ ]:
# -----------------------------
# Parameters
# -----------------------------
N = 600          # number of surface points
v0 = 1.0         # radial speed
nframes = 120    # animation frames
sigma = 0.08     # Gaussian width in velocity units
nv = 400         # velocity grid resolution

# Multipole coefficients (you can tweak these)
A0 = 1.0   # monopole
A1 = 0.7   # dipole amplitude
A2 = 0.5   # quadrupole amplitude

# Velocity grid for smoothed profile
v_grid = np.linspace(-v0, v0, nv)

# -----------------------------
# Generate random points on a sphere
# -----------------------------
phi = 2 * np.pi * np.random.rand(N)
mu = 2 * np.random.rand(N) - 1
theta = np.arccos(mu)

x = np.sqrt(1 - mu**2) * np.cos(phi)
y = np.sqrt(1 - mu**2) * np.sin(phi)
z = mu
r_hat = np.vstack((x, y, z)).T

# Line-of-sight direction: +z (default canvas orientation)
los = np.array([0, 0, 1])
v_los = v0 * (r_hat @ los)

# -----------------------------
# Multipole axis rotation (around y)
# -----------------------------
def multipole_axis(phi_rot):
    c, s = np.cos(phi_rot), np.sin(phi_rot)
    R = np.array([[ c, 0,  s],
                  [ 0, 1,  0],
                  [-s, 0,  c]])
    return R @ np.array([0, 0, 1])

# -----------------------------
# Gaussian smoothing per point
# -----------------------------
def smooth_profile(v_los, weights):
    diff = v_grid[:, None] - v_los[None, :]
    kernel = np.exp(-0.5 * (diff / sigma)**2)
    profile = kernel @ weights
    if np.max(profile) > 0:
        profile /= np.max(profile)
    return profile

# -----------------------------
# Figure setup
# -----------------------------
fig = plt.figure(figsize=(12, 5))
ax_sphere = fig.add_subplot(121, projection='3d')
ax_profile = fig.add_subplot(122)

# Sphere scatter
scatter = ax_sphere.scatter([], [], [], c=[], cmap="Blues", s=20)
quiver = None

# Orientation triad (bottom-left, default canvas axes)
triad_origin = np.array([-1.3, -1.3, -1.3])
triad_dirs = np.array([[0.4, 0,   0  ],   # +x
                       [0,   0.4, 0  ],   # +y
                       [0,   0,   0.4]])  # +z
triad_colors = ["red", "green", "blue"]
triad = ax_sphere.quiver(
    triad_origin[0], triad_origin[1], triad_origin[2],
    triad_dirs[:,0], triad_dirs[:,1], triad_dirs[:,2],
    length=0.4, normalize=False, colors=triad_colors, linewidth=2
)
# ax_sphere.text(-1.3+0.45, -1.3,      -1.3,      "x",      color="red")
# ax_sphere.text(-1.3,      -1.3+0.45, -1.3,      "y",      color="green")
# ax_sphere.text(0, 0, 1.1, "z (view)", color="blue")

# Line profile
line, = ax_profile.plot([], [], lw=2)

ax_sphere.set_title("l=0 to l=2 Sphere")
ax_sphere.set_xlim([-1.5, 1.5])
ax_sphere.set_ylim([-1.5, 1.5])
ax_sphere.set_zlim([-1.5, 1.5])
ax_sphere.set_box_aspect([1, 1, 1])

ax_profile.set_xlim([-v0, v0])
ax_profile.set_ylim([0, 1.1])
ax_profile.set_xlabel("v_los", y=0)
ax_profile.set_ylabel("Normalized intensity")

# -----------------------------
# Animation update
# -----------------------------
def update(frame):
    global quiver

    phi_rot = 2 * np.pi * frame / nframes
    axis = multipole_axis(phi_rot)

    # Project r_hat onto multipole axis
    dot = r_hat @ axis

    # Monopole + dipole + quadrupole weighting
    w_mono = A0 * np.ones_like(dot)
    w_dip  = A1 * dot
    w_quad = A2 * (3 * dot**2 - 1)
    w = w_mono + w_dip + w_quad

    # For coloring, normalize to [0,1]
    w_norm = (w - w.min()) / (w.max() - w.min())

    # Update sphere scatter
    scatter._offsets3d = (x, y, z)
    scatter.set_array(w_norm)

    # Update velocity arrows
    if quiver:
        quiver.remove()
    vectors = r_hat * w_norm[:, None]
    quiver = ax_sphere.quiver(
        x, y, z,
        vectors[:,0], vectors[:,1], vectors[:,2],
        length=0.2, normalize=False, color="black", linewidth=0.5
    )

    # Smoothed Doppler profile
    profile = smooth_profile(v_los, w)
    line.set_data(v_grid, profile)
    ax_profile.set_title(
        "Gaussian-Smoothed Line Profile"
    )

    return scatter, quiver, line, triad

anim = FuncAnimation(fig, update, frames=nframes, interval=80, blit=False)

plt.close()
HTML(anim.to_jshtml())


In [ ]:
%matplotlib widget


# -----------------------------
# Parameters
# -----------------------------
N = 600
v0 = 1.0
sigma = 0.08
nv = 400
v_grid = np.linspace(-v0, v0, nv)

# -----------------------------
# Generate random points on a sphere
# -----------------------------
phi = 2 * np.pi * np.random.rand(N)
mu = 2 * np.random.rand(N) - 1
theta = np.arccos(mu)

# x = np.sqrt(1 - mu**2) * np.cos(phi)
# y = np.sqrt(1 - mu**2) * np.sin(phi)
# z = mu
# r_hat = np.vstack((x, y, z)).T

def fibonacci_sphere(N):
    i = np.arange(N)
    phi = np.pi * (3. - np.sqrt(5.))  # golden angle
    y = 1 - 2*(i+0.5)/N
    r = np.sqrt(1 - y*y)
    theta = phi * i
    x = r * np.cos(theta)
    z = r * np.sin(theta)
    return np.vstack((x, y, z)).T

r_hat = fibonacci_sphere(N)
x, y, z = r_hat[:,0], r_hat[:,1], r_hat[:,2]

# LOS direction = +z
los = np.array([0, 0, 1])
v_los = v0 * (r_hat @ los)

# -----------------------------
# Gaussian smoothing per point
# -----------------------------
def smooth_profile(v_los, weights):
    diff = v_grid[:, None] - v_los[None, :]
    kernel = np.exp(-0.5 * (diff / sigma)**2)
    profile = kernel @ weights
    if np.max(profile) > 0:
        profile /= np.max(profile)
    return profile

# -----------------------------
# Multipole weighting
# -----------------------------
def compute_weights(A0, A1, A2):
    # Axis fixed along +z for GUI version
    axis = np.array([0, 0, 1])
    dot = r_hat @ axis

    w_mono = A0 * np.ones_like(dot)
    w_dip  = A1 * dot
    w_quad = A2 * (3 * dot**2 - 1)

    return w_mono + w_dip + w_quad

# -----------------------------
# Figure layout
# -----------------------------
fig = plt.figure(figsize=(12, 6))

ax_sphere = fig.add_subplot(121, projection='3d')
ax_profile = fig.add_subplot(122)

# Initial multipole strengths
A0_init, A1_init, A2_init = 1.0, 0.5, 0.3

# Initial weights
w = compute_weights(A0_init, A1_init, A2_init)
w_norm = (w - w.min()) / (w.max() - w.min())

# Sphere scatter
scatter = ax_sphere.scatter(x, y, z, c=w_norm, cmap="Blues", s=20)

# Velocity arrows
vectors = r_hat * w_norm[:, None]
quiver = ax_sphere.quiver(
    x, y, z,
    vectors[:,0], vectors[:,1], vectors[:,2],
    length=0.2, normalize=False, color="black", linewidth=0.5
)

# Orientation triad
triad_origin = np.array([-1.3, -1.3, -1.3])
triad_dirs = np.array([[0.4, 0, 0], [0, 0.4, 0], [0, 0, 0.4]])
triad_colors = ["red", "green", "blue"]
triad = ax_sphere.quiver(
    triad_origin[0], triad_origin[1], triad_origin[2],
    triad_dirs[:,0], triad_dirs[:,1], triad_dirs[:,2],
    length=0.4, normalize=False, colors=triad_colors, linewidth=2
)

# ax_sphere.text(-1.3+0.45, -1.3, -1.3, "x", color="red")
# ax_sphere.text(-1.3, -1.3+0.45, -1.3, "y", color="green")
# ax_sphere.text(-1.3, -1.3, -1.3+0.45, "z(view)", color="blue")

ax_sphere.set_xlim([-1.5, 1.5])
ax_sphere.set_ylim([-1.5, 1.5])
ax_sphere.set_zlim([-1.5, 1.5])
ax_sphere.set_box_aspect([1,1,1])
ax_sphere.set_title("Multipole Sphere")

# Initial line profile
profile = smooth_profile(v_los, w)
line, = ax_profile.plot(v_grid, profile, lw=2)
ax_profile.set_ylim([0, 1.1])
ax_profile.set_xlim([-v0, v0])
ax_profile.set_title("Gaussian-Smoothed Line Profile")

# -----------------------------
# Slider setup
# -----------------------------
axcolor = 'lightgoldenrodyellow'
ax_A0 = plt.axes([0.15, 0.02, 0.65, 0.03], facecolor=axcolor)
ax_A1 = plt.axes([0.15, 0.06, 0.65, 0.03], facecolor=axcolor)
ax_A2 = plt.axes([0.15, 0.10, 0.65, 0.03], facecolor=axcolor)

sA0 = Slider(ax_A0, 'Monopole A0', 0.0, 2.0, valinit=A0_init)
sA1 = Slider(ax_A1, 'Dipole A1', -1.0, 1.0, valinit=A1_init)
sA2 = Slider(ax_A2, 'Quadrupole A2', -1.0, 1.0, valinit=A2_init)

# -----------------------------
# Update function
# -----------------------------
def update(val):
    global quiver

    A0, A1, A2 = sA0.val, sA1.val, sA2.val
    w = compute_weights(A0, A1, A2)
    w_norm = (w - w.min()) / (w.max() - w.min())

    # Update sphere colors
    scatter.set_array(w_norm)

    # Update velocity arrows
    quiver.remove()
    vectors = r_hat * w_norm[:, None]
    quiver = ax_sphere.quiver(
        x, y, z,
        vectors[:,0], vectors[:,1], vectors[:,2],
        length=0.2, normalize=False, color="black", linewidth=0.5
    )

    # Update line profile
    profile = smooth_profile(v_los, w)
    line.set_ydata(profile)

    fig.canvas.draw_idle()

sA0.on_changed(update)
sA1.on_changed(update)
sA2.on_changed(update)

plt.show()


In [ ]:
%matplotlib widget


# ============================================================
# 1. Fibonacci sphere for uniform point distribution
# ============================================================
# ============================================================
# 2. Rotation matrices
# ============================================================
# ============================================================
# 3. Gaussian smoothing
# ============================================================
# ============================================================
# 4. Parameters
# ============================================================
N = 800
v0 = 1.0
sigma = 0.08
nv = 400
v_grid = np.linspace(-v0, v0, nv)

# Uniform sphere points
r_hat = fibonacci_sphere(N)

# LOS direction = +z
los = np.array([0, 0, 1])

# ============================================================
# 5. Multipole weighting
# ============================================================
def compute_weights(r, axis, A0, A1, A2):
    dot = r @ axis
    return A0 + A1*dot + A2*(3*dot**2 - 1)

# ============================================================
# 6. Figure layout
# ============================================================
fig = plt.figure(figsize=(12, 8))

# Move panels upward to create more space below
ax_sphere = fig.add_subplot(121, projection='3d')
ax_sphere.set_position([0.05, 0.32, 0.40, 0.63])

ax_profile = fig.add_subplot(122)
ax_profile.set_position([0.55, 0.32, 0.40, 0.63])

# Initial multipole strengths
A0_init, A1_init, A2_init = 1.0, 0.5, 0.3
rotX_init, rotY_init = 0.0, 0.0

# Initial multipole axis = +z
axis0 = np.array([0,0,1])

# Initial weights
w0 = compute_weights(r_hat, axis0, A0_init, A1_init, A2_init)
w_norm0 = (w0 - w0.min()) / (w0.max() - w0.min())

# Sphere scatter
scatter = ax_sphere.scatter(
    r_hat[:,0], r_hat[:,1], r_hat[:,2],
    c=w_norm0, cmap="Blues", s=20,
    vmin=0, vmax=1)

# Velocity arrows
vectors0 = r_hat * w_norm0[:, None]
quiver = ax_sphere.quiver(
    r_hat[:,0], r_hat[:,1], r_hat[:,2],
    vectors0[:,0], vectors0[:,1], vectors0[:,2],
    length=0.2, normalize=False, color="black", linewidth=0.5
)

# Orientation triad
triad_origin = np.array([-1.3, -1.3, -1.3])
triad_dirs = np.array([[0.6,0,0],[0,0.6,0],[0,0,0.6]])
triad_colors = ["red","green","blue"]
triad = ax_sphere.quiver(
    triad_origin[0], triad_origin[1], triad_origin[2],
    triad_dirs[:,0], triad_dirs[:,1], triad_dirs[:,2],
    length=0.4, normalize=False, colors=triad_colors, linewidth=2
)

# ax_sphere.text(-1.3+0.45, -1.3, -1.3, "x", color="red")
# ax_sphere.text(-1.3, -1.3+0.45, -1.3, "y", color="green")
# ax_sphere.text(-1.3, -1.3, -1.3+0.45, "z(view)", color="blue")

ax_sphere.set_xlim([-1.5,1.5])
ax_sphere.set_ylim([-1.5,1.5])
ax_sphere.set_zlim([-1.5,1.5])
ax_sphere.set_box_aspect([1,1,1])
ax_sphere.set_title("Multipole Sphere")

# Initial line profile
v_los0 = v0 * (r_hat @ los)
profile0 = smooth_profile(v_los0, w0, sigma, v_grid)
line, = ax_profile.plot(v_grid, profile0, lw=2)
ax_profile.set_ylim([0,1.1])
ax_profile.set_xlim([-v0,v0])
ax_profile.set_xlabel('Energy (keV)')
ax_profile.set_xticks((np.array([1140, 1150, 1160, 1170]) - 1157) / 20)
ax_profile.set_xticklabels(np.array([1140, 1150, 1160, 1170]))
ax_profile.set_title("Gaussian-Smoothed Line Profile")

# ============================================================
# 7. Sliders (now placed farther below the panels)
# ============================================================
axcolor = 'lightgoldenrodyellow'

# Sliders moved down to y = 0.20, 0.15, 0.10, 0.05
ax_A0 = plt.axes([0.10, 0.20, 0.35, 0.04], facecolor=axcolor)
ax_A1 = plt.axes([0.10, 0.15, 0.35, 0.04], facecolor=axcolor)
ax_A2 = plt.axes([0.10, 0.10, 0.35, 0.04], facecolor=axcolor)

ax_rotX = plt.axes([0.6, 0.20, 0.35, 0.04], facecolor=axcolor)
ax_rotY = plt.axes([0.6, 0.15, 0.35, 0.04], facecolor=axcolor)

sA0 = Slider(ax_A0, 'Monopole A0', 0.0, 2.0, valinit=A0_init)
sA1 = Slider(ax_A1, 'Dipole A1', -1.0, 1.0, valinit=A1_init)
sA2 = Slider(ax_A2, 'Quadrupole A2', -1.0, 1.0, valinit=A2_init)

sRotX = Slider(ax_rotX, 'Rotate X', -np.pi, np.pi, valinit=rotX_init)
sRotY = Slider(ax_rotY, 'Rotate Y', -np.pi, np.pi, valinit=rotY_init)

# ============================================================
# 8. Update function
# ============================================================
def update(val):
    global quiver

    A0, A1, A2 = sA0.val, sA1.val, sA2.val
    rotX, rotY = sRotX.val, sRotY.val

    # Rotation matrix
    R = R_y(rotY) @ R_x(rotX)

    # Rotate sphere points
    r_rot = (R @ r_hat.T).T
    x_r, y_r, z_r = r_rot[:,0], r_rot[:,1], r_rot[:,2]

    # Rotate multipole axis
    axis_rot = R @ axis0

    # LOS stays fixed along +z
    v_los_rot = v0 * (r_rot @ los)

    # Multipole weights
    w = compute_weights(r_rot, axis_rot, A0, A1, A2)
    w_norm = (w - w.min()) / (w.max() - w.min())

    # Update sphere
    scatter._offsets3d = (x_r, y_r, z_r)
    scatter.set_array(w_norm)
    scatter.set_clim(0, 1)

    # Update velocity arrows
    quiver.remove()
    vectors = r_rot * w_norm[:, None]
    quiver = ax_sphere.quiver(
        x_r, y_r, z_r,
        vectors[:,0], vectors[:,1], vectors[:,2],
        length=0.2, normalize=False, color="black", linewidth=0.5
    )

    # Update line profile
    profile = smooth_profile(v_los_rot, w, sigma, v_grid)
    line.set_ydata(profile)

    fig.canvas.draw_idle()

# Connect sliders
sA0.on_changed(update)
sA1.on_changed(update)
sA2.on_changed(update)
sRotX.on_changed(update)
sRotY.on_changed(update)

plt.show()


In [ ]:
%matplotlib widget


# ============================================================
# 1. Fibonacci sphere for uniform point distribution
# ============================================================
# ============================================================
# 2. Rotation matrices
# ============================================================
# ============================================================
# 3. Gaussian smoothing
# ============================================================
# ============================================================
# 4. Parameters
# ============================================================
N = 800
v0 = 1.0
sigma = 0.1125
nv = 400
v_grid = np.linspace(-v0, v0, nv)

# Uniform sphere points
r_hat = fibonacci_sphere(N)

# LOS direction = +z
los = np.array([0, 0, 1])

# ============================================================
# 5. Multipole weighting (quadrupole m=0 and m=2)
# ============================================================
def axis_frame(axis):
    """Build an orthonormal basis (e1, e2, e3) with e3 = axis."""
    e3 = axis / np.linalg.norm(axis)
    # pick a vector not parallel to e3
    if abs(e3[0]) < 0.9:
        tmp = np.array([1.0, 0.0, 0.0])
    else:
        tmp = np.array([0.0, 1.0, 0.0])
    e1 = np.cross(tmp, e3)
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(e3, e1)
    return e1, e2, e3

def compute_weights(r, axis, A0, A1, A2, frac_m2):
    """
    A2 is total quadrupole amplitude.
    frac_m2 in [0,1] sets how much of A2 goes into m=2 vs m=0:
      A20 = A2 * (1 - frac_m2)
      A22 = A2 * frac_m2
    """
    A20 = A2 * (1.0 - frac_m2)
    A22 = A2 * frac_m2

    e1, e2, e3 = axis_frame(axis)

    # coordinates in the multipole frame
    x_p = r @ e1
    y_p = r @ e2
    z_p = r @ e3

    phi = np.arctan2(y_p, x_p)

    # quadrupole components
    Q0 = 3.0 * z_p**2 - 1.0
    Q2 = (1.0 - z_p**2) * np.cos(2.0 * phi)

    return A0 + A1*z_p + A20*Q0 + A22*Q2

# ============================================================
# 6. Figure layout
# ============================================================
fig = plt.figure(figsize=(12, 8))

# Move panels upward to create more space below
ax_sphere = fig.add_subplot(121, projection='3d')
ax_sphere.set_position([0.05, 0.32, 0.40, 0.63])

ax_profile = fig.add_subplot(122)
ax_profile.set_position([0.55, 0.32, 0.40, 0.63])

# Initial multipole strengths
A0_init, A1_init, A2_init = 1.0, 0.5, 0.3
frac_m2_init = 0.0  # start with pure m=0
rotX_init, rotY_init = 0.0, 0.0

# Initial multipole axis = +z
axis0 = np.array([0,0,1])

# Initial weights
w0 = compute_weights(r_hat, axis0, A0_init, A1_init, A2_init, frac_m2_init)
w_norm0 = (w0 - w0.min()) / (w0.max() - w0.min())

# Sphere scatter
scatter = ax_sphere.scatter(
    r_hat[:,0], r_hat[:,1], r_hat[:,2],
    c=w_norm0, cmap="Blues", s=20,
    vmin=0, vmax=1)

# Velocity arrows
vectors0 = r_hat * w_norm0[:, None]
quiver = ax_sphere.quiver(
    r_hat[:,0], r_hat[:,1], r_hat[:,2],
    vectors0[:,0], vectors0[:,1], vectors0[:,2],
    length=0.2, normalize=False, color="black", linewidth=0.5
)

# Orientation triad
triad_origin = np.array([-1.3, -1.3, -1.3])
triad_dirs = np.array([[0.6,0,0],[0,0.6,0],[0,0,0.6]])
triad_colors = ["red","green","blue"]
triad = ax_sphere.quiver(
    triad_origin[0], triad_origin[1], triad_origin[2],
    triad_dirs[:,0], triad_dirs[:,1], triad_dirs[:,2],
    length=0.4, normalize=False, colors=triad_colors, linewidth=2
)

ax_sphere.set_xlim([-1.5,1.5])
ax_sphere.set_ylim([-1.5,1.5])
ax_sphere.set_zlim([-1.5,1.5])
ax_sphere.set_box_aspect([1,1,1])
ax_sphere.set_title("Multipole Sphere")

# Initial line profile
v_los0 = v0 * (r_hat @ los)
profile0 = smooth_profile(v_los0, w0, sigma, v_grid)
line, = ax_profile.plot(v_grid, profile0, lw=2)
ax_profile.set_ylim([0,1.1])
ax_profile.set_xlim([-v0,v0])
ax_profile.set_xlabel('Energy (keV)')
ax_profile.set_xticks((np.array([1140, 1150, 1160, 1170]) - 1157) / 20)
ax_profile.set_xticklabels(np.array([1140, 1150, 1160, 1170]))
ax_profile.set_title("Gaussian-Smoothed Line Profile")

# ============================================================
# 7. Sliders (now placed farther below the panels)
# ============================================================
axcolor = 'lightgoldenrodyellow'

ax_A0    = plt.axes([0.10, 0.10, 0.35, 0.04], facecolor=axcolor)
ax_A1    = plt.axes([0.10, 0.05, 0.35, 0.04], facecolor=axcolor)
ax_A2    = plt.axes([0.10, 0.00, 0.35, 0.04], facecolor=axcolor)
ax_frac2 = plt.axes([0.10, -0.05, 0.35, 0.04], facecolor=axcolor)

ax_rotX = plt.axes([0.6, 0.10, 0.35, 0.04], facecolor=axcolor)
ax_rotY = plt.axes([0.6, 0.05, 0.35, 0.04], facecolor=axcolor)

sA0    = Slider(ax_A0,    'Monopole A0', 0.0, 2.0, valinit=A0_init)
sA1    = Slider(ax_A1,    'Dipole A1',  -1.0, 1.0, valinit=A1_init)
sA2    = Slider(ax_A2,    'Quad A2',    -1.0, 1.0, valinit=A2_init)
sFrac2 = Slider(ax_frac2, 'Quad m2 frac', 0.0, 1.0, valinit=frac_m2_init)

sRotX = Slider(ax_rotX, 'Rotate X', -np.pi, np.pi, valinit=rotX_init)
sRotY = Slider(ax_rotY, 'Rotate Y', -np.pi, np.pi, valinit=rotY_init)

# ============================================================
# 8. Update function
# ============================================================
def update(val):
    global quiver

    A0    = sA0.val
    A1    = sA1.val
    A2    = sA2.val
    frac2 = sFrac2.val
    rotX  = sRotX.val
    rotY  = sRotY.val

    # Rotation matrix
    R = R_y(rotY) @ R_x(rotX)

    # Rotate sphere points
    r_rot = (R @ r_hat.T).T
    x_r, y_r, z_r = r_rot[:,0], r_rot[:,1], r_rot[:,2]

    # Rotate multipole axis
    axis_rot = R @ axis0

    # LOS stays fixed along +z
    v_los_rot = v0 * (r_rot @ los)

    # Multipole weights (with m=0 and m=2 quadrupole)
    w = compute_weights(r_rot, axis_rot, A0, A1, A2, frac2)
    w_norm = (w - w.min()) / (w.max() - w.min())

    # Update sphere
    scatter._offsets3d = (x_r, y_r, z_r)
    scatter.set_array(w_norm)
    scatter.set_clim(0, 1)

    # Update velocity arrows
    quiver.remove()
    vectors = r_rot * w_norm[:, None]
    quiver = ax_sphere.quiver(
        x_r, y_r, z_r,
        vectors[:,0], vectors[:,1], vectors[:,2],
        length=0.2, normalize=False, color="black", linewidth=0.5
    )

    # Update line profile
    profile = smooth_profile(v_los_rot, w, sigma, v_grid)
    line.set_ydata(profile)

    fig.canvas.draw_idle()

# Connect sliders
sA0.on_changed(update)
sA1.on_changed(update)
sA2.on_changed(update)
sFrac2.on_changed(update)
sRotX.on_changed(update)
sRotY.on_changed(update)

plt.show()
